<a href="https://colab.research.google.com/github/prksh830/Healthcare/blob/main/POA_AE_XGB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import numpy as np
import pandas as pd
import time

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef,
    roc_auc_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

import matplotlib.pyplot as plt


In [34]:
df = pd.read_csv("WSNBFSFdataset.csv")

for col in df.columns:
    print(repr(col))


'Event'
'Time'
'S_Node'
'Node_id'
'Rest_Energy'
'Trace_Level'
'Mac_Type_Pckt'
'Source_IP_Port'
'Des_IP_Port'
'Packet_Size'
'TTL'
'Hop_Count'
'Broadcast_ID'
'Dest_Node_Num'
'Dest_Seq_Num'
'Src_Node_ID'
'Src_Seq_Num'
'Class'


In [35]:
df = pd.read_csv("WSNBFSFdataset.csv")

# Strip spaces and hidden characters
df.columns = df.columns.str.strip()

print(df.columns.tolist())


['Event', 'Time', 'S_Node', 'Node_id', 'Rest_Energy', 'Trace_Level', 'Mac_Type_Pckt', 'Source_IP_Port', 'Des_IP_Port', 'Packet_Size', 'TTL', 'Hop_Count', 'Broadcast_ID', 'Dest_Node_Num', 'Dest_Seq_Num', 'Src_Node_ID', 'Src_Seq_Num', 'Class']


In [36]:
from sklearn.preprocessing import LabelEncoder

X = df.drop(columns=['Class']).values
y = LabelEncoder().fit_transform(df['Class'])


In [42]:
df = pd.read_csv("WSNBFSFdataset.csv")

X = df.drop(columns=['Class']).values
y = LabelEncoder().fit_transform(df['Class'])

# Identify and filter out classes with only one sample
class_counts = pd.Series(y).value_counts()
single_sample_classes = class_counts[class_counts == 1].index

if len(single_sample_classes) > 0:
    print(f"Warning: Classes with single samples detected: {single_sample_classes}. Removing these samples to enable stratification.")
    # Get indices of samples to keep
    indices_to_keep = pd.Series(y).apply(lambda label: label not in single_sample_classes)
    X = X[indices_to_keep.values]
    y = y[indices_to_keep.values]
    # Re-encode y after removing classes, if necessary (LabelEncoder will re-map 0, 1, 2...)
    y = LabelEncoder().fit_transform(y)

n_classes = len(np.unique(y))

In [38]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metrics = {
    'acc': [], 'macro_f1': [], 'macro_auc': [], 'mcc': []
}

start_total = time.time()


In [39]:
def poa_fitness(solution, Xtr, ytr):
    mask = solution[:-1] > 0.5
    if np.sum(mask) < 5:
        return -1

    lr = solution[-1]

    X_train, X_val, y_train, y_val = train_test_split(
        Xtr[:, mask], ytr,
        test_size=0.2,
        stratify=ytr,
        random_state=42
    )

    sample_weight = compute_sample_weight(
        class_weight='balanced',
        y=y_train
    )

    model = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=n_classes,
        n_estimators=150,
        max_depth=6,
        learning_rate=lr,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        tree_method='hist'
    )

    model.fit(X_train, y_train, sample_weight=sample_weight)

    probs = model.predict_proba(X_val)
    preds = np.argmax(probs, axis=1)

    macro_f1 = f1_score(y_val, preds, average='macro')
    macro_auc = roc_auc_score(y_val, probs, multi_class='ovr', average='macro')
    mcc = matthews_corrcoef(y_val, preds)

    feature_penalty = np.sum(mask) / Xtr.shape[1]

    return (0.35 * macro_f1 +
            0.35 * macro_auc +
            0.20 * mcc -
            0.10 * feature_penalty)


In [40]:
def POA(X, y, pop_size=8, iters=12):
    dim = X.shape[1] + 1
    population = np.random.rand(pop_size, dim)

    best_sol, best_fit = None, -np.inf

    for _ in range(iters):
        for i in range(pop_size):
            population[i][-1] = 0.01 + population[i][-1] * 0.2
            fit = poa_fitness(population[i], X, y)

            if fit > best_fit:
                best_fit = fit
                best_sol = population[i].copy()

        population += np.random.uniform(-0.15, 0.15, population.shape)
        population = np.clip(population, 0, 1)

    return best_sol


In [43]:
for fold, (tr, ts) in enumerate(skf.split(X, y), 1):
    print(f"\nFold {fold}")

    Xtr, Xts = X[tr], X[ts]
    ytr, yts = y[tr], y[ts]

    # Scaling
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xts = scaler.transform(Xts)

    # ===== Fold-wise Autoencoder =====
    input_dim = Xtr.shape[1]
    latent_dim = int(0.8 * input_dim)

    inp = Input(shape=(input_dim,))
    enc = Dense(latent_dim, activation='relu')(inp)
    dec = Dense(input_dim, activation='linear')(enc)

    autoencoder = Model(inp, dec)
    encoder = Model(inp, enc)

    autoencoder.compile(
        optimizer=Adam(0.001),
        loss='mse'
    )

    autoencoder.fit(
        Xtr, Xtr,
        epochs=30,
        batch_size=256,
        verbose=0
    )

    Xtr_enc = encoder.predict(Xtr)
    Xts_enc = encoder.predict(Xts)

    # ===== POA Optimization =====
    best = POA(Xtr_enc, ytr)
    mask = best[:-1] > 0.5
    lr = best[-1]

    sample_weight = compute_sample_weight(
        class_weight='balanced',
        y=ytr
    )

    model = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=n_classes,
        n_estimators=250,
        max_depth=6,
        learning_rate=lr,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        tree_method='hist'
    )

    model.fit(Xtr_enc[:, mask], ytr, sample_weight=sample_weight)

    probs = model.predict_proba(Xts_enc[:, mask])
    preds = np.argmax(probs, axis=1)

    metrics['acc'].append(accuracy_score(yts, preds))
    metrics['macro_f1'].append(f1_score(yts, preds, average='macro'))
    metrics['macro_auc'].append(
        roc_auc_score(yts, probs, multi_class='ovr', average='macro')
    )
    metrics['mcc'].append(matthews_corrcoef(yts, preds))



Fold 1
4430/4430 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Fold 2
4430/4430 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Fold 3
4430/4430 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Fold 4
4430/4430 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step  

Fold 5
4430/4430 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step  


In [44]:
print("\n===== POA-AE-XGB (4-Class WSN-BFSF) =====")
for k, v in metrics.items():
    print(f"{k.upper()}: {np.mean(v):.4f} ± {np.std(v):.4f}")

print(f"\nTotal Execution Time: {time.time() - start_total:.2f} sec")



===== POA-AE-XGB (4-Class WSN-BFSF) =====
ACC: 0.8630 ± 0.0102
MACRO_F1: 0.7000 ± 0.0122
MACRO_AUC: 0.9811 ± 0.0015
MCC: 0.7331 ± 0.0136

Total Execution Time: 3624.98 sec


In [45]:
cm = confusion_matrix(yts, preds)
print("\nConfusion Matrix:\n", cm)



Confusion Matrix:
 [[ 1084    95     0   120]
 [  115  5770    83     1]
 [    0    35   799    50]
 [ 3081     0  1378 22822]]
